# **AI Advanced Ahmed Yousrey Course Practice**

---
# **DAY 6 — Arabic Review Intelligence**
---

## STEP 1 — Importing Libraries

---

In [1]:
import pandas as pd
import numpy as np

import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB


from sklearn.metrics import classification_report, confusion_matrix

---
## STEP 2 — Load Data
---

In [2]:

df = pd.read_csv("/content/ar_reviews_100k.tsv", sep="\t")


---
## STEP 3 — EDA
---

In [3]:
df.shape

(99999, 2)

In [4]:
df.head()

,label,text
0,Positive,ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...
1,Positive,أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...
2,Positive,هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...
3,Positive,خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...
4,Positive,ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...


In [5]:
print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True) * 100)

label
Positive    33333
Mixed       33333
Negative    33333
Name: count, dtype: int64
label
Positive    33.333333
Mixed       33.333333
Negative    33.333333
Name: proportion, dtype: float64


In [6]:
df.isnull().sum()

,0
label,0
text,0


In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99999 entries, 0 to 99998
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   99999 non-null  object
 1   text    99999 non-null  object
dtypes: object(2)
memory usage: 1.5+ MB


get the number of characters description

In [9]:
df['text'].str.len().describe()

,text
count,99999.000000
mean,297.525285
std,514.015183
min,1.000000
25%,69.000000
50%,144.000000
75%,311.000000
max,8163.000000


get the number of words description

In [10]:
df['text'].str.split().str.len().describe()

,text
count,99999.000000
mean,55.026010
std,95.495538
min,1.000000
25%,13.000000
50%,26.000000
75%,57.000000
max,1663.000000


## EDA Interpretation — Text Length Analysis

**Right-skewed distribution:** The mean (297) is roughly double the median (144), which means most reviews are short but a small number of very long reviews pull the average up. The dataset is dominated by short-to-medium reviews.

**Minimum of 1:** Some reviews are a single character — possibly a lone punctuation mark or letter like "أ". After preprocessing (removing punctuation, stopwords, stemming), these will become empty strings. Empty strings passed to a vectorizer cause errors or silent noise. Fix: filter out rows where `clean_text` is empty after preprocessing.

**Maximum of 8,163:** TF-IDF handles length differences well by design — term frequency is computed within each document, so a long review is not unfairly weighted against a short one. The real risk with very long reviews is noise: more words means more off-topic content. This is not a breaking problem, but worth keeping in mind during error analysis.

---
## STEP 4 — PREPROCESSING
---

1.remove diacritics


In [11]:
# Keep a copy of the raw review so we can compare it with the cleaned version later.
df['original_text'] = df['text'].copy()

def remove_diacritics(text):
    arabic_diacritics = re.compile(r'[\u064B-\u0652\u0610-\u061A\u06DF-\u06E4\u0670\u06D6-\u06DC\u06DD\u06E5\u06E6]')
    return re.sub(arabic_diacritics, '', text)

df['text'] = df['text'].apply(remove_diacritics)
print("Diacritics removed from 'text' column.")
print(df.head())


Diacritics removed from 'text' column.
      label                                               text  \
0  Positive  ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...   
1  Positive  أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...   
2  Positive  هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...   
3  Positive  خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...   
4  Positive  ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...   

                                       original_text  
0  ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...  
1  أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...  
2  هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...  
3  خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...  
4  ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...  


2.normalize alef variants

In [12]:
def normalize_aleft(text):
    text = re.sub(r'[أإآ]', 'ا', text)
    return text

df['text'] = df['text'].apply(normalize_aleft)
print("Alef variants normalized in 'text' column.")
print(df.head())

Alef variants normalized in 'text' column.
      label                                               text  \
0  Positive  ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...   
1  Positive  احد اسباب نجاح الامارات ان كل شخص في هذه الدول...   
2  Positive  هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...   
3  Positive  خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...   
4  Positive  ياسات جلوريا جزء لا يتجزا من دبي . فندق متكامل...   

                                       original_text  
0  ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...  
1  أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...  
2  هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...  
3  خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...  
4  ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...  


3.remove non-arabic character

In [13]:
def remove_non_arabic(text):
    # Keep Arabic characters, numbers, and spaces. Remove all punctuation.
    text = re.sub(r'[^؀-ۿݐ-ݿࢠ-ࣿﭐ-﷏ﷰ-﷿ﹰ-﻿0-9\s]', '', text)
    return text

df['text'] = df['text'].apply(remove_non_arabic)
print("Non-Arabic characters and punctuation removed from 'text' column.")
print(df.head())

Non-Arabic characters and punctuation removed from 'text' column.
      label                                               text  \
0  Positive  ممتاز نوعا ما  النظافة والموقع والتجهيز والشاط...   
1  Positive  احد اسباب نجاح الامارات ان كل شخص في هذه الدول...   
2  Positive  هادفة  وقوية تنقلك من صخب شوارع القاهرة الى هد...   
3  Positive  خلصنا  مبدئيا اللي مستني ابهار زي الفيل الازرق...   
4  Positive  ياسات جلوريا جزء لا يتجزا من دبي  فندق متكامل ...   

                                       original_text  
0  ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...  
1  أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...  
2  هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...  
3  خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...  
4  ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...  


4.remove extra white space

In [14]:
def remove_extra_whitespace(text):
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(remove_extra_whitespace)
print("Extra white spaces removed from 'text' column.")
print(df.head())

Extra white spaces removed from 'text' column.
      label                                               text  \
0  Positive  ممتاز نوعا ما النظافة والموقع والتجهيز والشاطي...   
1  Positive  احد اسباب نجاح الامارات ان كل شخص في هذه الدول...   
2  Positive  هادفة وقوية تنقلك من صخب شوارع القاهرة الى هدو...   
3  Positive  خلصنا مبدئيا اللي مستني ابهار زي الفيل الازرق ...   
4  Positive  ياسات جلوريا جزء لا يتجزا من دبي فندق متكامل ا...   

                                       original_text  
0  ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...  
1  أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...  
2  هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...  
3  خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...  
4  ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...  


5.tokenize

In [15]:
def tokenize_text(text):
    return ' '.join(text.split())

df['text'] = df['text'].apply(tokenize_text)
print("Text tokenized in 'text' column.")
print(df.head())

Text tokenized in 'text' column.
      label                                               text  \
0  Positive  ممتاز نوعا ما النظافة والموقع والتجهيز والشاطي...   
1  Positive  احد اسباب نجاح الامارات ان كل شخص في هذه الدول...   
2  Positive  هادفة وقوية تنقلك من صخب شوارع القاهرة الى هدو...   
3  Positive  خلصنا مبدئيا اللي مستني ابهار زي الفيل الازرق ...   
4  Positive  ياسات جلوريا جزء لا يتجزا من دبي فندق متكامل ا...   

                                       original_text  
0  ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...  
1  أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...  
2  هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...  
3  خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...  
4  ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...  


6.remove stopwards

In [16]:
# Install NLTK if it is not already available, then download Arabic stopwords.
!pip install -q nltk

import nltk
nltk.download('stopwords', quiet=True)


True

In [17]:
from nltk.corpus import stopwords

# Get Arabic stopwords
arabic_stopwords = set(stopwords.words('arabic'))

def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in arabic_stopwords])

df['text'] = df['text'].apply(remove_stopwords)
print("Stopwords removed from 'text' column.")
print(df.head())

Stopwords removed from 'text' column.
      label                                               text  \
0  Positive  ممتاز نوعا النظافة والموقع والتجهيز والشاطيء ا...   
1  Positive  احد اسباب نجاح الامارات ان شخص الدولة يعشق ترا...   
2  Positive  هادفة وقوية تنقلك صخب شوارع القاهرة الى هدوء ج...   
3  Positive  خلصنا مبدئيا اللي مستني ابهار زي الفيل الازرق ...   
4  Positive  ياسات جلوريا جزء يتجزا دبي فندق متكامل الخدمات...   

                                       original_text  
0  ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...  
1  أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...  
2  هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...  
3  خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...  
4  ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...  


7.stem and rejoining tokens


In [18]:
from nltk.stem.isri import ISRIStemmer

st = ISRIStemmer()

def stem_text(text):
    return ' '.join([st.stem(word) for word in text.split()])

df['text'] = df['text'].apply(stem_text)
print("Text stemmed in 'text' column.")
print(df.head())

Text stemmed in 'text' column.
      label                                               text  \
0  Positive                      متز نوع نظف وقع جهز شاطيء طعم   
1  Positive  احد سبب نجح امر ان شخص دول عشق ترب نحب امر ومض...   
2  Positive  هدف وقي نقل صخب شرع قهر الى هدء جبل شيش عرف حق...   
3  Positive  خلص بدئ الل مست بهر زي فيل زرق ميقراش احس حمد ...   
4  Positive       ياس جلر جزء تجز دبي ندق كامل خدم ريح نفس وجد   

                                       original_text  
0  ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...  
1  أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...  
2  هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...  
3  خلصنا .. مبدئيا اللي مستني ابهار زي الفيل الاز...  
4  ياسات جلوريا جزء لا يتجزأ من دبي . فندق متكامل...  


### Demonstrating the Punctuation Fix

The `remove_non_arabic` function has been updated to remove all punctuation. To apply this fix to your DataFrame, please re-run the preprocessing cells from **cell `Ft54xbay9Z01` (remove non-arabic character)** down to **cell `Xv8kuyGR-FvZ` (stem and rejoining tokens)**.

Below, I'll show you a 'before' and 'after' for row 0 to illustrate the change, assuming you have re-run the cells.

In [19]:
# Show a real example before and after all preprocessing steps.
comparison = pd.DataFrame({
    'Before preprocessing': df['original_text'].head(3),
    'After preprocessing': df['text'].head(3),
    'Label': df['label'].head(3)
})

display(comparison)


,Before preprocessing,After preprocessing,Label
0,ممتاز نوعا ما . النظافة والموقع والتجهيز والشا...,متز نوع نظف وقع جهز شاطيء طعم,Positive
1,أحد أسباب نجاح الإمارات أن كل شخص في هذه الدول...,احد سبب نجح امر ان شخص دول عشق ترب نحب امر ومض...,Positive
2,هادفة .. وقوية. تنقلك من صخب شوارع القاهرة الى...,هدف وقي نقل صخب شرع قهر الى هدء جبل شيش عرف حق...,Positive


In [20]:
# Check for empty strings after preprocessing, then remove them before vectorization.
empty = df[df['text'].str.strip() == '']
print(f"Empty rows: {len(empty)}")

df = df[df['text'].str.strip() != ''].copy()
print(f"Remaining rows: {len(df)}")


Empty rows: 11
Remaining rows: 99988


---
**Step 5 — Train/Test Split**

----

In [21]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Why `stratify=y`?

`stratify=y` keeps the same label proportions in both the training and test sets. This prevents a minority label in this review dataset from being underrepresented—or missing—in the test set, which would make the evaluation unreliable.

### Why do we split before fitting the vectorizer?

The vectorizer learns its vocabulary from the text. It must be fitted on `X_train` only; otherwise, words from `X_test` would leak into the feature space and make the final score look better than it really is.


---
**STEP 6 — vectorizer**


---

In [22]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

print("Training matrix shape:", X_train_counts.shape)

Training matrix shape: (79990, 69127)


---
**STEP 7 — Model Training and Evaluation**
---

### 1. Multinomial Naive Bayes

In [23]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

mnb_model = MultinomialNB()
mnb_model.fit(X_train_counts, y_train)

mnb_predictions = mnb_model.predict(X_test_counts)

print("Multinomial Naive Bayes Classification Report:")
print(classification_report(y_test, mnb_predictions))

print("Multinomial Naive Bayes Confusion Matrix:")
print(confusion_matrix(y_test, mnb_predictions))

Multinomial Naive Bayes Classification Report:
              precision    recall  f1-score   support

       Mixed       0.53      0.48      0.51      6666
    Negative       0.63      0.70      0.66      6666
    Positive       0.67      0.65      0.66      6666

    accuracy                           0.61     19998
   macro avg       0.61      0.61      0.61     19998
weighted avg       0.61      0.61      0.61     19998

Multinomial Naive Bayes Confusion Matrix:
[[3211 1915 1540]
 [1347 4698  621]
 [1470  865 4331]]


### Naive Bayes Insight

Multinomial Naive Bayes is a fast baseline for word-count features. Focus on the F1-score for each label, not only accuracy, to see whether the model performs fairly across classes.


### 2. Logistic Regression

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_counts, y_train)

lr_predictions = lr_model.predict(X_test_counts)

print("Logistic Regression Classification Report:")
print(classification_report(y_test, lr_predictions))

print("Logistic Regression Confusion Matrix:")
print(confusion_matrix(y_test, lr_predictions))

Logistic Regression Classification Report:
              precision    recall  f1-score   support

       Mixed       0.54      0.52      0.53      6666
    Negative       0.67      0.68      0.68      6666
    Positive       0.65      0.67      0.66      6666

    accuracy                           0.62     19998
   macro avg       0.62      0.62      0.62     19998
weighted avg       0.62      0.62      0.62     19998

Logistic Regression Confusion Matrix:
[[3459 1515 1692]
 [1430 4547  689]
 [1486  714 4466]]


### Logistic Regression Insight

Logistic Regression is often a strong baseline for text classification. Compare its macro-average F1-score with Naive Bayes to judge whether it improves performance across all labels.


In [25]:
print(y_train.value_counts())
print(y_test.value_counts())

label
Mixed       26664
Positive    26663
Negative    26663
Name: count, dtype: int64
label
Mixed       6666
Positive    6666
Negative    6666
Name: count, dtype: int64


### 3. Linear Support Vector Machine (LinearSVC)

In [26]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

lsvc_model = LinearSVC(random_state=42)
lsvc_model.fit(X_train_counts, y_train)

lsvc_predictions = lsvc_model.predict(X_test_counts)

print("Linear SVC Classification Report:")
print(classification_report(y_test, lsvc_predictions))

print("Linear SVC Confusion Matrix:")
print(confusion_matrix(y_test, lsvc_predictions))

/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Linear SVC Classification Report:
              precision    recall  f1-score   support

       Mixed       0.55      0.50      0.52      6666
    Negative       0.65      0.67      0.66      6666
    Positive       0.63      0.66      0.65      6666

    accuracy                           0.61     19998
   macro avg       0.61      0.61      0.61     19998
weighted avg       0.61      0.61      0.61     19998

Linear SVC Confusion Matrix:
[[3329 1595 1742]
 [1347 4479  840]
 [1429  809 4428]]


### Linear SVC Insight

Linear SVC is commonly effective for sparse text vectors. Choose the best model based primarily on macro F1, then inspect the confusion matrix to understand its common mistakes.


---
## STEP 8 — Compare Models
---


In [27]:
from sklearn.metrics import accuracy_score, f1_score

results = pd.DataFrame({
    'Model': ['Multinomial Naive Bayes', 'Logistic Regression', 'Linear SVC'],
    'Accuracy': [
        accuracy_score(y_test, mnb_predictions),
        accuracy_score(y_test, lr_predictions),
        accuracy_score(y_test, lsvc_predictions)
    ],
    'Macro F1': [
        f1_score(y_test, mnb_predictions, average='macro'),
        f1_score(y_test, lr_predictions, average='macro'),
        f1_score(y_test, lsvc_predictions, average='macro')
    ]
}).sort_values('Macro F1', ascending=False)

print('Model Comparison:')
display(results)
print(f"Best model based on Macro F1: {results.iloc[0]['Model']}")


Model Comparison:


,Model,Accuracy,Macro F1
1,Logistic Regression,0.623662,0.622670
2,Linear SVC,0.611861,0.610017
0,Multinomial Naive Bayes,0.612061,0.609508


Best model based on Macro F1: Logistic Regression


### Final Project Insights

- A mean review length greater than the median indicates a **right-skewed** distribution: a few long reviews pull the mean upward.
- Reviews with one word should be checked after preprocessing because they may become empty or contain little useful signal.
- TF-IDF usually normalizes document vectors, so long documents do not automatically dominate; however, very long reviews can add noise and many extra features.
- Select the final baseline using **Macro F1**, especially if the labels are not perfectly balanced.
